# Mean SAE activations by race group

This notebook uses the 1,200-image open-prompt dataset (`a picture of a {race} {gender}`), not the 600-image engineered neutral-expression, T-shirt, and background set. It ranks the top ten features separately for each race by mean spatial activation, pools across the two gender groups, and compares each selected feature across all six race groups.

Feature IDs refer to the existing pretrained SAE. This is a descriptive prompt-group comparison; it does not establish feature meaning or causation. The full dataset is used for ranking and plotting, so this chart is exploratory rather than a held-out evaluation.

In [ ]:
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                     if p.name == 'sae-demo-notebooks' and (p / 'sae-activation').exists())
DEBIASING_ROOT = NOTEBOOK_ROOT.parent  # Edit this if the companion checkout is elsewhere.
FEATURES_PATH = DEBIASING_ROOT / 'backprop/sae_features.npz'
MANIFEST_PATH = DEBIASING_ROOT / 'backprop/manifest.csv'
TOP_K = 10

In [ ]:
# Confirm the manifest points at the open-prompt dataset.
with MANIFEST_PATH.open(newline='') as f:
    prompts = {row['prompt'] for row in csv.DictReader(f)}
assert prompts and all(p.startswith('a picture of a ') for p in prompts)
assert not any('t-shirt' in p or 'neutral white background' in p for p in prompts)

with np.load(FEATURES_PATH, allow_pickle=False) as data:
    X = data['X_mean'].astype(np.float64)
    race_ids = data['y_race']
    race_names = data['races'].tolist()

assert X.shape == (1200, 5120) and np.isfinite(X).all()
race_means = np.stack([X[race_ids == i].mean(axis=0) for i in range(len(race_names))])
top_features = [np.argsort(row)[-TOP_K:][::-1] for row in race_means]
print('Images per race group:', {name: int((race_ids == i).sum()) for i, name in enumerate(race_names)})

In [ ]:
colors = plt.get_cmap('tab10').colors
fig, axes = plt.subplots(2, 3, figsize=(19, 10), constrained_layout=True)
x = np.arange(TOP_K)
width = 0.13
for target_idx, (ax, target_race) in enumerate(zip(axes.flat, race_names)):
    features = top_features[target_idx]
    for compare_idx, compare_race in enumerate(race_names):
        offset = (compare_idx - (len(race_names) - 1) / 2) * width
        ax.bar(x + offset, race_means[compare_idx, features], width,
               label=compare_race.title(), color=colors[compare_idx])
    ax.set_title(f'Top 10 for {target_race.title()}')
    ax.set_xticks(x, [str(int(j)) for j in features], rotation=45, ha='right')
    ax.set_xlabel('SAE feature ID')
    ax.set_ylabel('Mean spatial activation')
    ax.grid(axis='y', alpha=0.25)
    ax.set_axisbelow(True)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncol=6, title='Race group')
fig.suptitle('Top 10 SAE features by mean activation on the open-prompt dataset\n'
             'Each panel ranks features within its title race; bars compare those features across races',
             fontsize=15)
plt.show()

In [ ]:
# Export a tidy table, including all race means for every selected feature.
output_path = NOTEBOOK_ROOT / 'sae-activation' / 'mean_activations_by_race.csv'
with output_path.open('w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ranking_race', 'rank', 'feature', 'mean_activation_on_ranking_race'] +
                    [f'mean_activation_{name.replace(" ", "_")}' for name in race_names])
    for i, race_name in enumerate(race_names):
        for rank, feature in enumerate(top_features[i], 1):
            writer.writerow([race_name, rank, int(feature), float(race_means[i, feature])] +
                            [float(race_means[j, feature]) for j in range(len(race_names))])
print(f'Wrote {output_path}')